# Dashboard failure mode 过滤验证

目标：验证 `src/dashboard` 可以把 `src/analysis` 导出的 failure mode 作为稳定筛选维度，用于 review 不同区间/regime 下的模型失败模式，而不是依赖 skill 或散落实验文件。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / 'SPEC.md').exists() and (REPO_ROOT / 'src').exists()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import DashboardViewFilters, build_dashboard_view_model, render_dashboard_html

EXP_DIR = Path('research/dashboard_failure_mode_filter_v1')
SOURCE_EXPORT = Path('research/dashboard_failure_modes_v1/dashboard_export.json')
SELECTED_FAILURE_MODE = 'return_stability_lag'

assert SOURCE_EXPORT.exists(), SOURCE_EXPORT
EXP_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
payload = json.loads(SOURCE_EXPORT.read_text(encoding='utf-8'))
full_view = build_dashboard_view_model(payload)
filtered_view = build_dashboard_view_model(
    payload,
    filters=DashboardViewFilters(failure_modes=(SELECTED_FAILURE_MODE,)),
)

failure_modes = payload.get('failure_modes', [])
filtered_cards = filtered_view.get('model_cards', [])
summary = {
    'source_export': str(SOURCE_EXPORT),
    'selected_failure_mode': SELECTED_FAILURE_MODE,
    'total_failure_modes': len(failure_modes),
    'available_failure_modes': sorted({row.get('failure_mode', '') for row in failure_modes}),
    'full_model_card_count': len(full_view.get('model_cards', [])),
    'filtered_model_card_count': len(filtered_cards),
    'filtered_models': [card.get('model') for card in filtered_cards],
    'filtered_segments': sorted({
        row.get('segment', '')
        for card in filtered_cards
        for row in card.get('failure_modes', [])
    }),
    'control_failure_modes': filtered_view.get('controls', {}).get('failure_modes', []),
}
summary


{'source_export': 'research/dashboard_failure_modes_v1/dashboard_export.json',
 'selected_failure_mode': 'return_stability_lag',
 'total_failure_modes': 4,
 'available_failure_modes': ['return_stability_lag'],
 'full_model_card_count': 2,
 'filtered_model_card_count': 1,
 'filtered_models': ['gcn_mse'],
 'filtered_segments': ['all',
  'stress',
  'stress_window',
  'stress_window::stress'],
 'control_failure_modes': ['return_stability_lag']}

In [3]:
view_path = EXP_DIR / 'view_model_return_stability_lag.json'
html_path = EXP_DIR / 'dashboard_failure_mode_filter.html'
summary_path = EXP_DIR / 'analysis_summary.json'

html = render_dashboard_html(filtered_view, title='SOXL Failure Mode Filter')
view_path.write_text(json.dumps(filtered_view, ensure_ascii=False, indent=2), encoding='utf-8')
html_path.write_text(html, encoding='utf-8')
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

assert 'failure-mode-filter' in html
assert 'data-failure-modes="return_stability_lag"' in html
assert SELECTED_FAILURE_MODE in filtered_view['controls']['failure_modes']
assert filtered_cards, 'failure mode 过滤后至少应保留一个模型卡片'
assert all(
    SELECTED_FAILURE_MODE in {row.get('failure_mode') for row in card.get('failure_modes', [])}
    for card in filtered_cards
)

print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "source_export": "research/dashboard_failure_modes_v1/dashboard_export.json",
  "selected_failure_mode": "return_stability_lag",
  "total_failure_modes": 4,
  "available_failure_modes": [
    "return_stability_lag"
  ],
  "full_model_card_count": 2,
  "filtered_model_card_count": 1,
  "filtered_models": [
    "gcn_mse"
  ],
  "filtered_segments": [
    "all",
    "stress",
    "stress_window",
    "stress_window::stress"
  ],
  "control_failure_modes": [
    "return_stability_lag"
  ]
}


## 中文结论

- 做了什么：验证 dashboard view model 和静态 HTML 页面可以直接消费 `src/analysis` 导出的 `failure_modes`，并通过 `failure-mode-filter` 对模型卡片进行筛选。
- 核心指标：源 export 包含 4 条 failure mode；选择 `return_stability_lag` 后保留 1 张相关模型卡片，控件列表包含该 failure mode。
- regime 差异：当前 failure mode 覆盖 `stress`、`stress_window`、`stress_window::stress` 和 `all`，用于定位模型在压力区间相对基准的收益/稳定性弱点。
- 是否晋升：晋升为 dashboard 交互框架能力；不代表任何交易策略晋升到 production。
- 风险：当前样例只有一种 failure mode，后续需要用更多模型和多区间 sidecar 检查组合过滤体验。
- 下一步：把 failure mode 与 window/regime/attribution 联动结果纳入页面视觉回归和 schedule export 通知摘要。